# Ceres Solver: Non-linear Least Squares

Ceres can solve bounds-constrained, robustified non-linear least squares
problems of the form

$$
\begin{split}
\min_{\mathbf{x}} &\quad \frac{1}{2}\sum_{i} \rho_i\!\left(\bigl\|f_i\!\left(x_{i_1}, \ldots ,x_{i_k}\right)\bigr\|^2\right) \\
\text{s.t.} &\quad l_j \le x_j \le u_j
\end{split}
$$

This notebook covers:

1. The Gauss–Newton / Levenberg–Marquardt theory underlying Ceres.
2. A curve-fitting example end to end (residual functors, parameter blocks, problem setup, manual cost evaluation).
3. A bundle-adjustment example (residuals, Jacobians, sparsity).


## 1. Gauss–Newton and Levenberg–Marquardt

### 1.1 The optimization problem

Minimize the **sum of squared residuals**:

$$
\min_x \|r(x)\|^2 = \min_x \sum_{i=1}^N r_i(x)^2
$$

where:

- $x$ is the parameter being optimized,
- $r_i(x)$ is the residual for the $i$-th observation, a function of $x$.

### 1.2 First-order Taylor expansion

The residual function $r_i(x)$ is generally nonlinear. Approximate it around
the current estimate $x_k$ using a first-order Taylor expansion:

$$
r_i(x) \approx r_i(x_k) + J_i(x_k)\,(x - x_k)
$$

where:

- $r_i(x_k)$ is the residual at the current estimate,
- $J_i(x_k) = \dfrac{\partial r_i}{\partial x}$ is the Jacobian of the residual w.r.t. $x$.

### 1.3 Quadratic approximation of the cost

The cost function is approximated locally as

$$
\|r(x)\|^2 \approx \|r(x_k) + J(x_k)\,(x - x_k)\|^2
$$

where $r(x)$ is the vector of all residuals and $J(x)$ is the Jacobian matrix.
Expanding:

$$
\|r(x_k) + J(x_k)\,(x - x_k)\|^2
= \|r(x_k)\|^2 + 2\,(x - x_k)^\top J(x_k)^\top r(x_k) + (x - x_k)^\top J(x_k)^\top J(x_k)\,(x - x_k)
$$

### 1.4 Gauss–Newton update rule

Differentiate the quadratic approximation with respect to $x$ and set to zero:

$$
\nabla \|r(x)\|^2 = 2\,J(x_k)^\top r(x_k) + 2\,J(x_k)^\top J(x_k)\,(x - x_k) = 0
$$

Simplifying:

$$
J(x_k)^\top J(x_k)\,(x - x_k) = -J(x_k)^\top r(x_k)
$$

Solving for the new $x$:

$$
x_{k+1} = x_k - \bigl(J(x_k)^\top J(x_k)\bigr)^{-1} J(x_k)^\top r(x_k)
$$

This is the **Gauss–Newton update rule**.

### 1.5 Key terms in the update

| Term | Meaning |
| --- | --- |
| $J(x_k)$ | Jacobian of residuals w.r.t. parameters |
| $J(x_k)^\top J(x_k)$ | Approximate Hessian of the cost |
| $J(x_k)^\top r(x_k)$ | Gradient of the cost |
| $\bigl(J^\top J\bigr)^{-1} J^\top r$ | Step direction |

### 1.6 Limitations of Gauss–Newton

- **Convergence**: Gauss–Newton works well when residuals are small and the
  cost is close to quadratic. For highly nonlinear problems or large
  residuals it may fail to converge.
- **Hessian approximation**: $J^\top J$ ignores second-order terms, which can
  be inaccurate for highly nonlinear problems.

### 1.7 Damped Gauss–Newton: Levenberg–Marquardt

Ceres Solver typically uses **Levenberg–Marquardt**, which combines
Gauss–Newton with a damping term:

$$
\bigl(J^\top J + \lambda I\bigr)\,(x_{k+1} - x_k) = -J^\top r(x_k)
$$

The scalar $\lambda$ controls the damping:

- small $\lambda$ → behaves like Gauss–Newton,
- large $\lambda$ → behaves more like gradient descent, ensuring stability.


## 2. Curve-Fitting Example

Solve the curve-fitting problem $y = e^{m x + c}$ in Ceres without applying
logarithmic transformations, following the
[Ceres Solver tutorial](http://ceres-solver.org/nnls_tutorial.html#curve-fitting)
and the
[curve fitting example](https://ceres-solver.googlesource.com/ceres-solver/+/master/examples/curve_fitting.cc).

### 2.1 Problem setup

**Equation**

$$
y_i = e^{m x_i + c}
$$

**Residual to minimize**

$$
r_i = y_i - e^{m x_i + c}
$$

**Definitions**

1. **ResidualBlock**
   - One residual block per data point $(x_i, y_i)$.
   - $f_i(m, c) = y_i - e^{m x_i + c}$.
2. **CostFunction**
   - For a single residual: $\text{CostFunction}(m, c) = y_i - e^{m x_i + c}$.
3. **ParameterBlock**
   - The parameter block contains the parameters being optimized, $m$ and $c$
     (size $2$ if packed into one block, size $1$ each if passed separately).
4. **Index $i$** – data-point index, $i = 1, 2, \ldots, 10$.
5. **Index $k$** – number of parameters, $k = 2$ ($m$, $c$).


### 2.2 Residual functor with a packed parameter block

```cpp
struct ExponentialResidualPackedParams {
  ExponentialResidualPackedParams(double x, double y) : x_(x), y_(y) {}

  template <typename T>
  bool operator()(const T *const params, T *residual) const {
    residual[0] = T(y_) - exp(params[0] * T(x_) + params[1]);
    // params[0] = m, params[1] = c
    return true;
  }

private:
  const double x_;
  const double y_;
};
```

```text
auto* cost_function = new AutoDiffCostFunction<ExponentialResidualPackedParams, 1, 2>(...);
                                                                                ^  ^
                                                                                |  |
   Dimension of residual --------------------------------------------------------+  |
   Dimension of parameter block ({m, c}) ------------------------------------------+
```

**Components**

1. **Dimension of residual** — size of the residual vector $r_i$ per data
   point. For $r_i = y_i - e^{m x_i + c}$ the residual is scalar, so the
   dimension is $1$.
2. **Dimension of parameter block** — size of the parameter block being
   optimized. Here the block is $[m, c]$, so the dimension is $2$.


### 2.3 Residual functor with separate parameter blocks

```cpp
struct ExponentialResidualSeparatedParams {
  ExponentialResidualSeparatedParams(double x, double y) : x_(x), y_(y) {}

  template <typename T>
  bool operator()(const T *const m, const T *const c, T *residual) const {
    residual[0] = T(y_) - exp(m[0] * T(x_) + c[0]);
    return true;
  }

private:
  const double x_;
  const double y_;
};
```

```text
auto* cost_function = new AutoDiffCostFunction<ExponentialResidualSeparatedParams, 1, 1, 1>(...);
                                                                                   ^  ^  ^
                                                                                   |  |  |
   Dimension of residual ---------------------------------------------------------+  |  |
   Dimension of parameter m  --------------------------------------------------------+  |
   Dimension of parameter c  -----------------------------------------------------------+
```

**Components**

1. **Dimension of residual** — $1$ (scalar residual).
2. **Dimension of parameter $m$** — $1$.
3. **Dimension of parameter $c$** — $1$.


### 2.4 Adding the cost function to a `Problem`

Initial estimates for the parameters and the loop that registers one residual
block per data point:

```cpp
double params[2] = {0.0, 0.0};  // Initial guess for [m, c]

ceres::Problem problem;
for (size_t i = 0; i < x_data.size(); ++i) {
  auto *cost_function =
      new ceres::AutoDiffCostFunction<ExponentialResidualPackedParams, 1, 2>(
          new ExponentialResidualPackedParams(x_data[i], y_data[i]));
  problem.AddResidualBlock(cost_function, nullptr, params);
}
```


### 2.5 Manually evaluating the cost function

The functor's `operator()` is invoked indirectly by the `CostFunction` class
during optimization. It is also possible to call `Evaluate` directly to
inspect the residual and Jacobian at a given point:

```cpp
double x = 2.0;
double y = exp(2.0 * x + 1.0);  // True value of y for m=2.0, c=1.0

// Define the residual functor
auto *cost_function =
    new ceres::AutoDiffCostFunction<ExponentialResidualPackedParams, 1, 2>(
        new ExponentialResidualPackedParams(x, y));

// Parameters at which to evaluate the cost function
double params[2] = {1.5, 0.5};   // Initial guess: m=1.5, c=0.5
double residuals[1];             // To store residuals
double *jacobians[1];            // To store Jacobians

double jacobian_values[2];
jacobians[0] = jacobian_values;

const double *params_ptr[] = {params};
bool success = cost_function->Evaluate(params_ptr, residuals, jacobians);

if (success) {
  std::cout << "Residual: "        << residuals[0] << "\n";
  std::cout << "Jacobian wrt m: "  << jacobians[0][0] << "\n";
  std::cout << "Jacobian wrt c: "  << jacobians[0][1] << "\n";
} else {
  std::cout << "Cost function evaluation failed.\n";
}
```


### 2.6 Output and Jacobian shape

- **Residual**: $r_i = y - e^{m x + c}$, evaluated at the given parameters.
- **Jacobian**:
  - **Packed parameter block** (Section 2.2): the Jacobian is a $1 \times 2$
    matrix $\bigl[\,\tfrac{\partial r_i}{\partial m},\ \tfrac{\partial r_i}{\partial c}\bigr]$.
  - **Separate parameter blocks** (Section 2.3): each block's Jacobian is
    $1 \times 1$:
    - $\dfrac{\partial r_i}{\partial m}$ for $m$,
    - $\dfrac{\partial r_i}{\partial c}$ for $c$.

**When to use which form**

- Use the **packed** form when $m$ and $c$ are optimized as a single block.
- Use the **separate** form when $m$ and $c$ are independent parameter blocks
  (e.g. for setting bounds or constancy on each individually).


## 3. Bundle Adjustment Example

Setup: **5 points in 3D space** and **3 cameras**, where each 3D point is
visible in some subset of the cameras.

### 3.1 Problem setup

1. **Cameras.** Three cameras with a known camera model (e.g., a pinhole
   model with intrinsic parameters). Each camera has a pose (rotation
   $\mathbf{R}$ and translation $\mathbf{t}$), forming a 6D vector per camera:

$$
\text{camera parameters: } \mathbf{x}_{c_i} = [\mathbf{R}_i,\, \mathbf{t}_i]
$$

2. **3D points.** Five 3D points, each $\mathbf{p}_j = [X_j, Y_j, Z_j]$:

$$
\text{point parameters: } \mathbf{x}_{p_j} = [X_j, Y_j, Z_j]
$$

3. **Observations.** For each visible point in a camera there is a 2D
   observation $(u_{ij}, v_{ij})$ — the projection of the 3D point onto that
   camera's image plane.

4. **Residuals.** The residual for one observation is the difference between
   the observed pixel and the projected pixel:

$$
f_{ij}(\mathbf{x}_{c_i}, \mathbf{x}_{p_j}) =
\begin{bmatrix}
u_{ij} - \hat{u}_{ij} \\
v_{ij} - \hat{v}_{ij}
\end{bmatrix}
$$

   where $(\hat{u}_{ij}, \hat{v}_{ij})$ is the projection of $\mathbf{p}_j$
   into camera $i$.

5. **Optimization problem.** Minimize the sum of squared residuals over all
   observations:

$$
\min_{\mathbf{x}} \quad \frac{1}{2} \sum_{(i,j) \in \text{observations}}
\bigl\| f_{ij}(\mathbf{x}_{c_i}, \mathbf{x}_{p_j}) \bigr\|^2
$$

   where $\mathbf{x}$ groups all camera and 3D-point parameters.

### 3.2 Key terms

- **$\mathbf{x}$** — optimization variables: $\mathbf{x}_{c_i}$ (camera) and
  $\mathbf{x}_{p_j}$ (3D point).
- **$f_{ij}$** — residual function: difference between observed and projected
  pixel locations.
- **$k$** — each residual depends on $k = 9$ variables: $6$ from the camera,
  $3$ from the point.
- **Constraints** — optional bounds may be added on point positions or camera
  poses.

### 3.3 Worked observation pattern

- 3 cameras with parameters $\mathbf{x}_{c_1}, \mathbf{x}_{c_2}, \mathbf{x}_{c_3}$.
- 5 points $\mathbf{x}_{p_1}, \ldots, \mathbf{x}_{p_5}$.
- Observation list:
  - Camera 1 observes $\mathbf{x}_{p_1}, \mathbf{x}_{p_2}, \mathbf{x}_{p_3}$.
  - Camera 2 observes $\mathbf{x}_{p_2}, \mathbf{x}_{p_3}, \mathbf{x}_{p_4}$.
  - Camera 3 observes $\mathbf{x}_{p_3}, \mathbf{x}_{p_5}$.

### 3.4 Camera model (pinhole projection)

$$
\begin{bmatrix}
\hat{u}_{ij} \\
\hat{v}_{ij}
\end{bmatrix}
=
\begin{bmatrix}
f_x \dfrac{X'_j}{Z'_j} + c_x \\[4pt]
f_y \dfrac{Y'_j}{Z'_j} + c_y
\end{bmatrix}
$$

where:

- $(X'_j, Y'_j, Z'_j) = \mathbf{R}_i\,\mathbf{p}_j + \mathbf{t}_i$ is the
  point in the camera's coordinate system,
- $(f_x, f_y)$ are the focal lengths,
- $(c_x, c_y)$ are the principal-point offsets.

Each residual depends on the parameters of one camera ($6$) and one 3D point
($3$), giving $k = 9$ inputs per residual.


### 3.5 Residuals across all observations

Each residual corresponds to the reprojection error of a 3D point into a 2D
observation. For each observation $(i, j)$:

- $i$ — camera index,
- $j$ — point index,
- $(u_{ij}, v_{ij})$ — observed pixel,
- $(\hat{u}_{ij}, \hat{v}_{ij})$ — predicted pixel from the camera model.

$$
f_{ij}(\mathbf{x}_{c_i}, \mathbf{x}_{p_j}) =
\begin{bmatrix}
u_{ij} - \hat{u}_{ij} \\
v_{ij} - \hat{v}_{ij}
\end{bmatrix}
$$

The predicted coordinates are computed by:

1. **Transform the 3D point into the camera frame**:

$$
\begin{bmatrix} X'_j \\ Y'_j \\ Z'_j \end{bmatrix}
= \mathbf{R}_i \mathbf{p}_j + \mathbf{t}_i
$$

2. **Project onto the image plane**:

$$
\hat{u}_{ij} = f_x \frac{X'_j}{Z'_j} + c_x, \qquad
\hat{v}_{ij} = f_y \frac{Y'_j}{Z'_j} + c_y
$$

#### Full objective

$$
\min_{\mathbf{x}} \quad \frac{1}{2} \sum_{(i, j)}
\bigl\| f_{ij}(\mathbf{x}_{c_i}, \mathbf{x}_{p_j}) \bigr\|^2
$$

with $\mathbf{x} = \{\mathbf{x}_{c_i}, \mathbf{x}_{p_j}\}$ collecting all
camera and point parameters.

#### Example expansion (3 cameras, 5 points)

Observations:

- Camera 1 observes $p_1, p_2$.
- Camera 2 observes $p_2, p_3, p_4$.
- Camera 3 observes $p_3, p_5$.

Residuals:

**Camera 1**

$$
f_{11}(\mathbf{x}_{c_1}, \mathbf{x}_{p_1}) =
\begin{bmatrix} u_{11} - \hat{u}_{11} \\ v_{11} - \hat{v}_{11} \end{bmatrix},
\qquad
f_{12}(\mathbf{x}_{c_1}, \mathbf{x}_{p_2}) =
\begin{bmatrix} u_{12} - \hat{u}_{12} \\ v_{12} - \hat{v}_{12} \end{bmatrix}
$$

**Camera 2**

$$
f_{22}(\mathbf{x}_{c_2}, \mathbf{x}_{p_2}) =
\begin{bmatrix} u_{22} - \hat{u}_{22} \\ v_{22} - \hat{v}_{22} \end{bmatrix},
\quad
f_{23}(\mathbf{x}_{c_2}, \mathbf{x}_{p_3}) =
\begin{bmatrix} u_{23} - \hat{u}_{23} \\ v_{23} - \hat{v}_{23} \end{bmatrix},
\quad
f_{24}(\mathbf{x}_{c_2}, \mathbf{x}_{p_4}) =
\begin{bmatrix} u_{24} - \hat{u}_{24} \\ v_{24} - \hat{v}_{24} \end{bmatrix}
$$

**Camera 3**

$$
f_{33}(\mathbf{x}_{c_3}, \mathbf{x}_{p_3}) =
\begin{bmatrix} u_{33} - \hat{u}_{33} \\ v_{33} - \hat{v}_{33} \end{bmatrix},
\qquad
f_{35}(\mathbf{x}_{c_3}, \mathbf{x}_{p_5}) =
\begin{bmatrix} u_{35} - \hat{u}_{35} \\ v_{35} - \hat{v}_{35} \end{bmatrix}
$$

#### Compact form

$$
f_{ij}(\mathbf{x}_{c_i}, \mathbf{x}_{p_j}) =
\begin{bmatrix}
u_{ij} - \left(f_x \dfrac{X'_j}{Z'_j} + c_x\right) \\[4pt]
v_{ij} - \left(f_y \dfrac{Y'_j}{Z'_j} + c_y\right)
\end{bmatrix},
\qquad
\begin{bmatrix} X'_j \\ Y'_j \\ Z'_j \end{bmatrix}
= \mathbf{R}_i \mathbf{p}_j + \mathbf{t}_i
$$

Each residual depends on $k = 9$ variables ($6$ from the camera, $3$ from the
point), exactly as in Section 3.2.


### 3.6 Jacobian for a single residual

Each residual $f_{ij}$ contributes a Jacobian with two parts:

$$
\mathbf{J}_{ij} =
\begin{bmatrix}
\dfrac{\partial f_{ij}}{\partial \mathbf{x}_{c_i}} &
\dfrac{\partial f_{ij}}{\partial \mathbf{x}_{p_j}}
\end{bmatrix}
$$

#### 3.6.1 Derivatives w.r.t. camera parameters

Camera parameters typically include rotation $\mathbf{R}_i$ (parameterized as
a 3D vector via Rodrigues, axis-angle, etc.) and translation $\mathbf{t}_i$.

1. Transformed 3D point:

$$
\begin{bmatrix} X'_j \\ Y'_j \\ Z'_j \end{bmatrix}
= \mathbf{R}_i \mathbf{p}_j + \mathbf{t}_i
$$

2. Derivatives w.r.t. translation $\mathbf{t}_i$:

$$
\frac{\partial X'_j}{\partial \mathbf{t}_i} = \begin{bmatrix} 1 & 0 & 0 \end{bmatrix}, \quad
\frac{\partial Y'_j}{\partial \mathbf{t}_i} = \begin{bmatrix} 0 & 1 & 0 \end{bmatrix}, \quad
\frac{\partial Z'_j}{\partial \mathbf{t}_i} = \begin{bmatrix} 0 & 0 & 1 \end{bmatrix}
$$

3. Derivatives w.r.t. rotation $\mathbf{R}_i$ depend on the parameterization
   (Rodrigues, quaternions, etc.) and are obtained via the chain rule:

$$
\frac{\partial X'_j}{\partial \mathbf{R}_i}, \quad
\frac{\partial Y'_j}{\partial \mathbf{R}_i}, \quad
\frac{\partial Z'_j}{\partial \mathbf{R}_i}
$$

4. Pinhole projection derivatives:

$$
\frac{\partial \hat{u}_{ij}}{\partial X'_j} = \frac{f_x}{Z'_j}, \quad
\frac{\partial \hat{u}_{ij}}{\partial Y'_j} = 0, \quad
\frac{\partial \hat{u}_{ij}}{\partial Z'_j} = -f_x \frac{X'_j}{(Z'_j)^2}
$$

$$
\frac{\partial \hat{v}_{ij}}{\partial X'_j} = 0, \quad
\frac{\partial \hat{v}_{ij}}{\partial Y'_j} = \frac{f_y}{Z'_j}, \quad
\frac{\partial \hat{v}_{ij}}{\partial Z'_j} = -f_y \frac{Y'_j}{(Z'_j)^2}
$$

5. Combine via the chain rule to assemble $\dfrac{\partial f_{ij}}{\partial \mathbf{x}_{c_i}}$.

#### 3.6.2 Derivatives w.r.t. 3D point parameters

The point parameters are $[X_j, Y_j, Z_j]$.

1. Transformed point as in 3.6.1.
2. Derivatives of the transformed point:

$$
\frac{\partial X'_j}{\partial \mathbf{p}_j} = \mathbf{R}_i[0, :], \quad
\frac{\partial Y'_j}{\partial \mathbf{p}_j} = \mathbf{R}_i[1, :], \quad
\frac{\partial Z'_j}{\partial \mathbf{p}_j} = \mathbf{R}_i[2, :]
$$

   where $\mathbf{R}_i[k, :]$ is the $k$-th row of the rotation matrix.

3. Use the same pinhole-projection derivatives from 3.6.1 to compute
   $\dfrac{\partial \hat{u}_{ij}}{\partial \mathbf{p}_j}$ and
   $\dfrac{\partial \hat{v}_{ij}}{\partial \mathbf{p}_j}$.

4. Combine to assemble $\dfrac{\partial f_{ij}}{\partial \mathbf{x}_{p_j}}$.

#### 3.6.3 Per-observation Jacobian shape

For a single observation, $\mathbf{J}_{ij}$ has shape $2 \times (6 + 3) = 2
\times 9$:

- residual $f_{ij}$ is $2$-D ($u, v$),
- camera parameters are $6$-D (rotation + translation),
- 3D-point parameters are $3$-D ($[X_j, Y_j, Z_j]$).

#### 3.6.4 Full Jacobian shape and sparsity

For the full bundle-adjustment problem:

- The Jacobian has shape
  $\bigl(2 \cdot \text{num\_observations}\bigr) \times \bigl(\text{num\_camera\_params} + \text{num\_point\_params}\bigr)$.
- It is sparse, because each residual depends only on the parameters of one
  camera and one point.

If camera $c_1$ observes point $p_2$, the corresponding block is

$$
\mathbf{J}_{(1,2)} =
\begin{bmatrix}
\dfrac{\partial f_{12}}{\partial \mathbf{x}_{c_1}} &
\dfrac{\partial f_{12}}{\partial \mathbf{x}_{p_2}}
\end{bmatrix}
$$

placed at the appropriate row/column locations of the sparse global Jacobian.
The sparsity is what makes large-scale BA tractable.


#### 3.6.5 Why angle-axis is a typical choice

In Bundler / BAL-style cost functions (such as Snavely's, used in many of the
Ceres BA examples), the rotation $\mathbf{R}_i$ is parameterized by an
angle-axis 3-vector $\boldsymbol{\omega}$ rather than a quaternion. The
practical reasons:

- **Unconstrained 3-D vector** — Levenberg–Marquardt updates it directly,
  whereas quaternions require renormalization or a `Manifold` /
  `LocalParameterization` to enforce $\|\mathbf{q}\| = 1$.
- **Smaller per-camera block** — 6 DOF (rotation + translation), or 9 DOF if
  intrinsics are part of the block, instead of 7 / 10 with a quaternion.
- **Plays well with `AutoDiffCostFunction`** — `ceres::AngleAxisRotatePoint`
  is templated on the scalar type, so the parameter block stays a flat
  contiguous array.
- **No practical singularity** — the $\theta = \pi$ wrap is far from any LM
  step, and Ceres switches to a Taylor expansion near $\theta = 0$.

Quaternions are preferable when per-step rotations are large without a warm
start, when SLERP between keyframes is needed, or in pipelines that already
operate on SO(3) on the manifold (GTSAM, Sophus, manif). See
[noah_snavely_reprojection_error.ipynb §2.7](noah_snavely_reprojection_error.ipynb)
for a longer discussion.


### 3.7 Worked Jacobian: 2 cameras, 2 points, 3 observations

A small case that makes the sparsity pattern explicit.

**Setup**

- Cameras ($C$): $c_1, c_2$ — so $C = 2$.
- Points ($P$): $p_1, p_2$ — so $P = 2$.
- Observations ($N = 3$):
  - $c_1$ observes $p_1$,
  - $c_1$ observes $p_2$,
  - $c_2$ observes $p_1$.

Each observation contributes 2 residuals ($u$ and $v$), so the Jacobian has
$2N = 6$ rows.

**Jacobian dimensions**

$$
\mathbf{J} \in \mathbb{R}^{6 \times (6C + 3P)} = \mathbb{R}^{6 \times (12 + 6)} = \mathbb{R}^{6 \times 18}
$$

Each residual depends on:

- $6$ camera parameters: rotation $(r_x, r_y, r_z)$ and translation $(t_x, t_y, t_z)$ of the observing camera,
- $3$ point parameters: $X_j, Y_j, Z_j$ of the observed 3D point.

The Jacobian is sparse: a row block is only non-zero in the columns of the
camera that made the observation and the point that was observed.

#### Per-observation residuals and Jacobian blocks

**Observation 1** ($c_1$ observes $p_1$)

$$
f_{11} =
\begin{bmatrix} u_{11} - \hat{u}_{11} \\ v_{11} - \hat{v}_{11} \end{bmatrix},
\qquad
\mathbf{J}_{11} =
\begin{bmatrix}
\dfrac{\partial \hat{u}_{11}}{\partial \mathbf{x}_{c_1}} & \dfrac{\partial \hat{u}_{11}}{\partial \mathbf{x}_{p_1}} \\[6pt]
\dfrac{\partial \hat{v}_{11}}{\partial \mathbf{x}_{c_1}} & \dfrac{\partial \hat{v}_{11}}{\partial \mathbf{x}_{p_1}}
\end{bmatrix}
$$

**Observation 2** ($c_1$ observes $p_2$)

$$
f_{12} =
\begin{bmatrix} u_{12} - \hat{u}_{12} \\ v_{12} - \hat{v}_{12} \end{bmatrix},
\qquad
\mathbf{J}_{12} =
\begin{bmatrix}
\dfrac{\partial \hat{u}_{12}}{\partial \mathbf{x}_{c_1}} & \dfrac{\partial \hat{u}_{12}}{\partial \mathbf{x}_{p_2}} \\[6pt]
\dfrac{\partial \hat{v}_{12}}{\partial \mathbf{x}_{c_1}} & \dfrac{\partial \hat{v}_{12}}{\partial \mathbf{x}_{p_2}}
\end{bmatrix}
$$

**Observation 3** ($c_2$ observes $p_1$)

$$
f_{21} =
\begin{bmatrix} u_{21} - \hat{u}_{21} \\ v_{21} - \hat{v}_{21} \end{bmatrix},
\qquad
\mathbf{J}_{21} =
\begin{bmatrix}
\dfrac{\partial \hat{u}_{21}}{\partial \mathbf{x}_{c_2}} & \dfrac{\partial \hat{u}_{21}}{\partial \mathbf{x}_{p_1}} \\[6pt]
\dfrac{\partial \hat{v}_{21}}{\partial \mathbf{x}_{c_2}} & \dfrac{\partial \hat{v}_{21}}{\partial \mathbf{x}_{p_1}}
\end{bmatrix}
$$

#### Full Jacobian (block-sparse)

Splitting each $\mathbf{J}_{ij}$ into camera-side $\mathbf{J}_{ij}^{(c)}$ and
point-side $\mathbf{J}_{ij}^{(p)}$ pieces, the full $6 \times 18$ Jacobian is

$$
\mathbf{J} =
\begin{bmatrix}
\mathbf{J}_{11}^{(c_1)} & 0                  & \mathbf{J}_{11}^{(p_1)} & 0 \\
\mathbf{J}_{12}^{(c_1)} & 0                  & 0                       & \mathbf{J}_{12}^{(p_2)} \\
0                       & \mathbf{J}_{21}^{(c_2)} & \mathbf{J}_{21}^{(p_1)} & 0
\end{bmatrix}
$$

where:

- each $\mathbf{J}_{ij}^{(c)}$ is $2 \times 6$ (camera parameters),
- each $\mathbf{J}_{ij}^{(p)}$ is $2 \times 3$ (point parameters),
- column blocks are: camera 1 (6), camera 2 (6), point 1 (3), point 2 (3).

#### Explicit per-element derivatives

Recall
$\hat{u}_{ij} = f_x \dfrac{X'_j}{Z'_j} + c_x$,
$\hat{v}_{ij} = f_y \dfrac{Y'_j}{Z'_j} + c_y$,
with $(X'_j, Y'_j, Z'_j)^\top = \mathbf{R}_i \mathbf{p}_j + \mathbf{t}_i$.

**Camera-translation derivatives**

$$
\frac{\partial \hat{u}_{ij}}{\partial t_x} = \frac{f_x}{Z'_j}, \qquad
\frac{\partial \hat{v}_{ij}}{\partial t_x} = 0
$$

$$
\frac{\partial \hat{u}_{ij}}{\partial t_y} = 0, \qquad
\frac{\partial \hat{v}_{ij}}{\partial t_y} = \frac{f_y}{Z'_j}
$$

$$
\frac{\partial \hat{u}_{ij}}{\partial t_z} = -f_x \frac{X'_j}{(Z'_j)^2}, \qquad
\frac{\partial \hat{v}_{ij}}{\partial t_z} = -f_y \frac{Y'_j}{(Z'_j)^2}
$$

**Camera-rotation derivatives**

These are more involved and depend on the rotation parameterization. Denote
them generically as
$\dfrac{\partial \hat{u}_{ij}}{\partial r_x},
\dfrac{\partial \hat{u}_{ij}}{\partial r_y},
\dfrac{\partial \hat{u}_{ij}}{\partial r_z},$ and similarly for $\hat{v}_{ij}$.

**3D-point derivatives** (chain rule, $\mathbf{R}_i[a, b]$ denotes the rotation
matrix entry):

$$
\frac{\partial \hat{u}_{ij}}{\partial X_j} = \frac{f_x}{Z'_j}\,\mathbf{R}_i[0, 0], \quad
\frac{\partial \hat{u}_{ij}}{\partial Y_j} = \frac{f_x}{Z'_j}\,\mathbf{R}_i[0, 1], \quad
\frac{\partial \hat{u}_{ij}}{\partial Z_j} = \frac{f_x}{Z'_j}\,\mathbf{R}_i[0, 2]
$$

with analogous expressions for
$\dfrac{\partial \hat{v}_{ij}}{\partial X_j},
\dfrac{\partial \hat{v}_{ij}}{\partial Y_j},
\dfrac{\partial \hat{v}_{ij}}{\partial Z_j}$.

#### Putting it together

Substituting the values of $(X'_j, Y'_j, Z'_j)$ and the derivatives above
fills in each $2 \times 9$ block $\mathbf{J}_{ij}$ — the first 6 columns are
camera-parameter derivatives, the last 3 columns are point-parameter
derivatives. Placing each block into the global sparse Jacobian at the right
camera/point columns produces the full $\mathbf{J}$ used in the Gauss–Newton
or Levenberg–Marquardt step from Section 1.
